# 02 — Preprocessing

## Input
`data/processed/clinvar_filtered.tsv.gz` — 383,533 variants (output of 01_eda)

## Steps
1. Filter to SNVs only (341,826 rows, 89.1% of filtered dataset)
2. Drop anomalous rows (1 row with allele length > 1, 1 row with na allele)
3. One-hot encode ref and alt alleles → 8-dimensional vector per variant
4. Stratified train/val/test split (70/10/20)

## Output
Saved to `data/processed/`:
- `X_train.npy` — (239,277, 8) — 14.6MB
- `X_val.npy` — (34,183, 8) — 2.1MB
- `X_test.npy` — (68,365, 8) — 4.2MB
- `y_train.npy`, `y_val.npy`, `y_test.npy` — labels for each split

## Key decisions
- SNV-only: variable length alleles deferred — see dissertation/confounders.md
- Stratified split: preserves 4.6:1

In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split

sys.path.append(str(Path().resolve().parent))
import config

# Load filtered dataset
df = pd.read_csv(
    config.PROCESSED_DIR / "clinvar_filtered.tsv.gz",
    sep='\t',
    low_memory=False
)

print(f"Loaded: {df.shape}")
print(f"Label distribution:\n{df['Label'].value_counts()}")

Loaded: (383533, 44)
Label distribution:
Label
0    288464
1     95069
Name: count, dtype: int64


In [2]:
# Filter to SNVs only
df_snv = df[df['Type'] == 'single nucleotide variant'].copy()

# Drop the one row with allele length > 1
df_snv = df_snv[df_snv['ReferenceAlleleVCF'].str.len() == 1]
df_snv = df_snv[df_snv['AlternateAlleleVCF'].str.len() == 1]

# Drop the one row with na allele
df_snv = df_snv[df_snv['ReferenceAlleleVCF'] != 'na']
df_snv = df_snv[df_snv['AlternateAlleleVCF'] != 'na']

# Reset index
df_snv = df_snv.reset_index(drop=True)

print(f"SNV shape: {df_snv.shape}")
print(f"Label distribution:\n{df_snv['Label'].value_counts()}")
print(f"Class ratio: {df_snv['Label'].value_counts()[0]/df_snv['Label'].value_counts()[1]:.1f}:1")

SNV shape: (341825, 44)
Label distribution:
Label
0    280866
1     60959
Name: count, dtype: int64
Class ratio: 4.6:1


In [3]:
# One-hot encoding for DNA bases
NUCLEOTIDES = ['A', 'C', 'G', 'T']

def one_hot_encode(base):
    """Encode a single nucleotide as a 4-element one-hot vector."""
    vector = [0, 0, 0, 0]
    if base in NUCLEOTIDES:
        vector[NUCLEOTIDES.index(base)] = 1
    return vector

def encode_variant(ref, alt):
    """
    Encode a SNV as a concatenated one-hot vector.
    Returns a list of length 8: [ref_onehot, alt_onehot]
    """
    return one_hot_encode(ref) + one_hot_encode(alt)

# Apply encoding
print("Encoding variants...")
X = np.array([
    encode_variant(ref, alt)
    for ref, alt in zip(df_snv['ReferenceAlleleVCF'], df_snv['AlternateAlleleVCF'])
])
y = df_snv['Label'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nExample encodings:")
for i in range(3):
    ref = df_snv['ReferenceAlleleVCF'].iloc[i]
    alt = df_snv['AlternateAlleleVCF'].iloc[i]
    print(f"  {ref}→{alt}: {encode_variant(ref, alt)}")

Encoding variants...
X shape: (341825, 8)
y shape: (341825,)

Example encodings:
  C→T: [0, 1, 0, 0, 0, 0, 0, 1]
  A→C: [1, 0, 0, 0, 0, 1, 0, 0]
  G→A: [0, 0, 1, 0, 1, 0, 0, 0]


In [4]:
print("All possible SNV encodings:")
for ref in NUCLEOTIDES:
    for alt in NUCLEOTIDES:
        if ref != alt:
            print(f"  {ref}→{alt}: {encode_variant(ref, alt)}")

All possible SNV encodings:
  A→C: [1, 0, 0, 0, 0, 1, 0, 0]
  A→G: [1, 0, 0, 0, 0, 0, 1, 0]
  A→T: [1, 0, 0, 0, 0, 0, 0, 1]
  C→A: [0, 1, 0, 0, 1, 0, 0, 0]
  C→G: [0, 1, 0, 0, 0, 0, 1, 0]
  C→T: [0, 1, 0, 0, 0, 0, 0, 1]
  G→A: [0, 0, 1, 0, 1, 0, 0, 0]
  G→C: [0, 0, 1, 0, 0, 1, 0, 0]
  G→T: [0, 0, 1, 0, 0, 0, 0, 1]
  T→A: [0, 0, 0, 1, 1, 0, 0, 0]
  T→C: [0, 0, 0, 1, 0, 1, 0, 0]
  T→G: [0, 0, 0, 1, 0, 0, 1, 0]


In [5]:
# Stratified split — preserves class ratio across all three sets
# First split off test set (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_SEED,
    stratify=y
)

# Then split remaining into train and val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=config.VAL_SIZE / (1 - config.TEST_SIZE),
    random_state=config.RANDOM_SEED,
    stratify=y_train_val
)

print(f"Train:      {X_train.shape} | Labels: {np.bincount(y_train)}")
print(f"Validation: {X_val.shape} | Labels: {np.bincount(y_val)}")
print(f"Test:       {X_test.shape} | Labels: {np.bincount(y_test)}")

# Confirm class ratios are preserved
for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    ratio = np.bincount(y_split)[0] / np.bincount(y_split)[1]
    print(f"{name} class ratio: {ratio:.1f}:1")

Train:      (239277, 8) | Labels: [196606  42671]
Validation: (34183, 8) | Labels: [28087  6096]
Test:       (68365, 8) | Labels: [56173 12192]
Train class ratio: 4.6:1
Val class ratio: 4.6:1
Test class ratio: 4.6:1


In [6]:
# Save splits to processed directory
np.save(config.PROCESSED_DIR / "X_train.npy", X_train)
np.save(config.PROCESSED_DIR / "X_val.npy", X_val)
np.save(config.PROCESSED_DIR / "X_test.npy", X_test)
np.save(config.PROCESSED_DIR / "y_train.npy", y_train)
np.save(config.PROCESSED_DIR / "y_val.npy", y_val)
np.save(config.PROCESSED_DIR / "y_test.npy", y_test)

print("Splits saved:")
for name in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test']:
    path = config.PROCESSED_DIR / f"{name}.npy"
    size_mb = path.stat().st_size / 1024 / 1024
    print(f"  {name}.npy — {size_mb:.1f} MB")

Splits saved:
  X_train.npy — 14.6 MB
  X_val.npy — 2.1 MB
  X_test.npy — 4.2 MB
  y_train.npy — 1.8 MB
  y_val.npy — 0.3 MB
  y_test.npy — 0.5 MB
